# knot — 01: deploy the **v1** spec (movies domain)

Walk-through arc:

1. **01_deploy** (this notebook) — deploy the v1 spec
2. 02_ingest — ingest imdb movies
3. 03_migrate — compose in games + podcasts + tv + webscraped,
   run Atlas to apply
4. 04_ingest_more — ingest the 55 bindings the migration enabled
5. 05_er — compute Movie embeddings, run ER, watch the resolver fill

The spec is the package `movies_spec/`, structured like a
FastAPI app:

| file | what it owns |
|---|---|
| `base.py`       | the `Spec` object — three lines, no classes |
| `person.py`     | the shared `Person` class + all its slots |
| `movies.py`     | Movie + MovieCredit + DirectedMovie + 3 movie sources |
| `games.py`      | Studio + Platform + Game + Release + GameCredit + 3 srcs |
| `podcasts.py`   | Podcast + PodcastEpisode + PodcastCredit + 3 srcs |
| `tv.py`         | Show + Season + TVEpisode + TVCredit + tvdb + cross-bind |
| `webscraped.py` | Mention class + 4 low-trust scraper sources |
| `full.py`       | composes the migration-time domains onto v1 |

Loading `movies_spec` gives you the v1: `base + person + movies`.
The other domains stay opt-in until 03_migrate imports
`movies_spec.full`.

In [2]:
import pandas as pd
from _demo import SCHEMA, connect

In [3]:
# ``from movies_spec import spec`` → spec object with the BASE
# entities only. movies_spec.full has not been imported, so the
# spec doesn't know about tmdb or the embedding slot yet.
from movies_spec import spec

print("classes:", list(spec.classes))
print("sources:", list(spec.sources))
print("Movie slots:", [s.name for s in spec.classes["Movie"].slots])
spec

classes: ['Person', 'Movie', 'MovieCredit', 'DirectedMovie']
sources: ['imdb', 'tmdb', 'rottentomatoes']
Movie slots: ['canonical_id', 'title', 'year', 'director', 'runtime_minutes', 'title_embedding']


Spec(identifier_slot_name='canonical_id', schema='knot_demo', classes={'Person': OntologyClass(name='Person', kind=<ClassKind.CONCRETE: 'concrete'>, is_a=None, mixins=[], slots=[Slot(name='canonical_id', type=<Primitive.TEXT: 'text'>, identifier=True, required=True, description=None), Slot(name='name', type=<Primitive.TEXT: 'text'>, identifier=False, required=True, description=None), Slot(name='birth_country', type=<Primitive.TEXT: 'text'>, identifier=False, required=False, description=None), Slot(name='birth_year', type=<Primitive.INTEGER: 'integer'>, identifier=False, required=False, description=None), Slot(name='role_description', type=<Primitive.TEXT: 'text'>, identifier=False, required=False, description=None), Slot(name='role_summary', type=<Primitive.TEXT: 'text'>, identifier=False, required=False, description=None), Slot(name='name_embedding', type=Vector(dim=384, metric='cosine'), identifier=False, required=False, description=None)], description=None), 'Movie': OntologyClass(n

In [4]:
# ``_demo.connect()`` returns (psycopg, sqlalchemy engine). SCHEMA
# is the fixed throwaway schema name every notebook uses; they
# chain. 01 drops + recreates so re-runs are clean.
pg, engine = connect()
pg.execute(f"DROP SCHEMA IF EXISTS {SCHEMA} CASCADE")

<psycopg.Cursor [COMMAND_OK] [IDLE] (host=localhost port=5433 database=knot) at 0x7d09ad1159d0>

In [5]:
# The canonical target schema for the base spec, as one SQL script.
# Nothing executes yet — just the text. Notice the order: schema →
# weight table → canonical tables → bindings tables → indexes →
# FK alters → resolved views → all-sources views.
#
# No CREATE EXTENSION vector — the base spec has no vector slot.
print(spec.ddl())

CREATE SCHEMA IF NOT EXISTS knot_demo;

CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS knot_demo.source_weight (
    source_name text NOT NULL,
    class_name  text NOT NULL,
    slot_name   text NOT NULL,
    weight      double precision NOT NULL,
    PRIMARY KEY (source_name, class_name, slot_name)
);

CREATE TABLE IF NOT EXISTS knot_demo.person (
    canonical_id text NOT NULL,
    name text NOT NULL,
    birth_country text,
    birth_year integer,
    role_description text,
    role_summary text,
    name_embedding vector(384),
    PRIMARY KEY (canonical_id)
);

CREATE INDEX IF NOT EXISTS person_name_embedding_hnsw_idx
    ON knot_demo.person
    USING hnsw (name_embedding vector_cosine_ops);

CREATE TABLE IF NOT EXISTS knot_demo.person_bindings (
    source_name text NOT NULL,
    source_identifier text NOT NULL,
    canonical_id text,
    name text,
    birth_country text,
    birth_year integer,
    role_description text,
    role_summary text,
    name_embed

In [6]:
# First deploy against an empty schema: execute directly. Every
# statement is idempotent (CREATE TABLE IF NOT EXISTS / CREATE OR
# REPLACE VIEW), so re-running is a no-op.
pg.execute(spec.ddl())

<psycopg.Cursor [COMMAND_OK] [IDLE] (host=localhost port=5433 database=knot) at 0x7d09ad1162d0>

In [7]:
# What landed?
pd.read_sql_query(
    """
    SELECT table_name, table_type
    FROM information_schema.tables
    WHERE table_schema = %(schema)s
    UNION ALL
    SELECT viewname, 'VIEW'
    FROM pg_views WHERE schemaname = %(schema)s
    ORDER BY 2, 1
    """,
    engine,
    params={"schema": SCHEMA},
)

,table_name,table_type
0,movie,BASE TABLE
1,movie_bindings,BASE TABLE
2,moviecredit,BASE TABLE
3,moviecredit_bindings,BASE TABLE
4,person,BASE TABLE
5,person_bindings,BASE TABLE
6,source_weight,BASE TABLE
7,directedmovie,VIEW
8,directedmovie,VIEW
9,movie_all_sources,VIEW
